# Plot one specific DOM waveform

Give it a `(run, event, DOM)` triple, it fetches the calibrated ATWD + fADC waveform via icetray (subprocess) and renders the plot. Tweak the plotting cell at will — fetch only runs once per target, then you can iterate on the figure.

In [ ]:
import subprocess, textwrap, tempfile, pickle, os, sys, re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# ---------------- Plot settings (Overleaf/LaTeX-ready) ----------------
matplotlib.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.unicode_minus": False,
    "pgf.rcfonts": False,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})


def export_to_pdf(fig, filename):
    # NB: no bbox_inches='tight' here — keep file size == fig figsize exactly.
    fig.savefig(filename, format="pdf", pad_inches=0)


ENV_SHELL = '/cvmfs/icecube.opensciencegrid.org/py3-v4.3.0/RHEL_9_x86_64/metaprojects/icetray/v1.11.1/env-shell.sh'

# ---------------- Data source ----------------
DATA_SAMPLE = 'MINIONS'   # 'MINIONS' or 'BURNSAMPLE'

if DATA_SAMPLE == 'MINIONS':
    MINION_I3  = '/lustre/hpc/icecube/janikh/MINIONS_DA_sample_2015_v5.i3.zst'
    MINION_GCD = '/lustre/hpc/icecube/janikh/GeoCalibDetectorStatus_2015.57161_V0.i3.gz'
    DATA_SOURCE_LABEL = r'IceCube IC86.2015 burnsample'
    def files_for_run(run_id):
        return [MINION_GCD, MINION_I3]
else:
    BURN_ROOT = Path('/lustre/hpc/project/icecube/Burnsample/I3files/IC86.22')
    GCD_DIR   = BURN_ROOT / 'GCD'
    DATA_SOURCE_LABEL = r'IceCube IC86.2022 burnsample'
    def files_for_run(run_id):
        gcd = next(iter(GCD_DIR.glob(
            f'Level2_IC86.2022_data_Run{run_id:08d}_*_GCD.i3.zst')), None)
        if gcd is None:
            raise FileNotFoundError(f'no GCD for run {run_id}')
        subs = sorted(BURN_ROOT.glob(
            f'oscNext_data_IC86.22_*_Run{run_id:08d}_Subrun*.i3.zst'))
        if not subs:
            raise FileNotFoundError(f'no subrun files for run {run_id}')
        return [str(gcd)] + [str(p) for p in subs]


def run_in_icetray(python_code, timeout=3600):
    tmp_py  = Path(tempfile.mkstemp(suffix='.py')[1])
    tmp_pkl = Path(tempfile.mkstemp(suffix='.pkl')[1])
    tmp_py.write_text(textwrap.dedent(python_code))
    proc = subprocess.Popen(
        [ENV_SHELL, 'python', '-u', str(tmp_py)],
        env={**os.environ, 'OUT_PICKLE': str(tmp_pkl), 'PYTHONUNBUFFERED': '1'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line.rstrip()); sys.stdout.flush()
    proc.wait(timeout=timeout)
    if proc.returncode != 0:
        raise RuntimeError(f'icetray subprocess failed (rc={proc.returncode})')
    return tmp_pkl

## 1. Set your target
Edit these three values.

In [ ]:
TARGET_RUN   = 126491
TARGET_EVENT = 30343391
TARGET_OM    = (83, 31, 0)   # (string, om, pmt)

## 2. Fetch the calibrated waveform from the I3 file
Slow (one I3-pass with `I3WaveCalibrator`). Only re-run when you change the target.

In [ ]:
fetch_files = files_for_run(TARGET_RUN)
files_repr = repr(fetch_files)
print(f'fetching from run {TARGET_RUN}: {len(fetch_files)-1} subrun files')

fetch_code = f'''
    import os, math, pickle
    from icecube import icetray, dataio, dataclasses, WaveCalibrator
    from icecube.icetray import I3Tray, I3Units

    icetray.logging.set_level_for_unit("I3WaveCalibrator", "FATAL")

    FE_R, E_CHG = 50.0, 1.602176634e-19
    TARGET_RUN, TARGET_EVENT = {TARGET_RUN}, {TARGET_EVENT}
    TARGET_OM = {TARGET_OM!r}
    out = [None]

    def grab(frame):
        if out[0]: return
        if "InIceRawData" not in frame or "CalibratedWaveforms" not in frame: return
        hdr = frame["I3EventHeader"]
        if hdr.run_id != TARGET_RUN or hdr.event_id != TARGET_EVENT: return
        rd, cal_wfs = frame["InIceRawData"], frame["CalibratedWaveforms"]
        cal, det = frame["I3Calibration"], frame["I3DetectorStatus"]
        for om, launches in rd:
            om_t = (int(om.string), int(om.om), int(om.pmt))
            if om_t != TARGET_OM: continue
            L = next((x for x in launches if x.lc_bit), None)
            if L is None or om not in cal_wfs: continue
            if om not in cal.dom_cal or om not in det.dom_status: continue
            dc, ds = cal.dom_cal[om], det.dom_status[om]
            hv = float(ds.pmt_hv) / I3Units.V
            gain = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
            kf = 1.0 / (FE_R * gain * E_CHG)
            out[0] = {{
                "om": om_t,
                "pmt_gain": float(gain),
                "pmt_hv_volts": hv,
                "pe_per_voltsecond": kf,
                "hlc_launch_time_ns": float(L.time),
                "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                "calibrated_waveforms": [
                    {{"source": str(w.source), "channel": int(w.channel),
                      "time_ns": float(w.time), "bin_width_ns": float(w.bin_width),
                      "samples_volt": [float(v) / I3Units.V for v in w.waveform]}}
                    for w in cal_wfs[om]
                ],
            }}
            break

    def drop_existing(frame):
        for k in ("CalibratedWaveforms", "CalibrationErrata"):
            if k in frame: del frame[k]

    def has_launches(frame):
        return "InIceRawData" in frame

    def stop(frame):
        if out[0]: tray.RequestSuspension()

    tray = I3Tray()
    tray.Add("I3Reader", FilenameList={files_repr})
    tray.Add(drop_existing, Streams=[icetray.I3Frame.DAQ])
    tray.Add("I3WaveCalibrator", If=has_launches,
             Launches="InIceRawData",
             Waveforms="CalibratedWaveforms",
             WaveformRange="CalibratedWaveformRange_recalc")
    tray.Add(grab, Streams=[icetray.I3Frame.DAQ])
    tray.Add(stop, Streams=[icetray.I3Frame.DAQ])
    tray.Execute()

    if out[0] is None:
        raise RuntimeError(f"event {{TARGET_RUN}}/{{TARGET_EVENT}} DOM {{TARGET_OM}} not found")
    with open(os.environ["OUT_PICKLE"], "wb") as f:
        pickle.dump(out[0], f, protocol=pickle.HIGHEST_PROTOCOL)
    ev = out[0]["event"]
    print(f"fetched DOM {{out[0]['om']}}  run={{ev['run_id']}} event={{ev['event_id']}}")
'''

fp = run_in_icetray(fetch_code)
with open(fp, 'rb') as f:
    hit = pickle.load(f)
print(f"hit ready — gain={hit['pmt_gain']:.2e}, HV={hit['pmt_hv_volts']:.0f} V, "
      f"t_main={hit['hlc_launch_time_ns']:.0f} ns")

## 3. Plot — tweak this cell freely
Re-run this cell as much as you like; it doesn't touch the I3 file.

In [ ]:
# --- plotting knobs ---
# Saved file size == FIGSIZE exactly (no tight-bbox cropping).
FIGSIZE        = (5.8, 4.8)
LINEWIDTH      = 1.2
ATWD_COLOR     = 'C0'
FADC_COLOR     = 'C1'

XLIM_ATWD      = None
XLIM_FADC      = None
EDGE_PAD_FRAC  = 0.02

SAVE_PDF = True
SAVE_PNG = True
DPI      = 200
PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)

k = hit['pe_per_voltsecond']
t_main = hit['hlc_launch_time_ns']

atwd_all = [w for w in hit['calibrated_waveforms'] if w['source'] == 'ATWD']
fadc_all = [w for w in hit['calibrated_waveforms'] if w['source'] == 'FADC']

def near_t_main(wfs):
    if not wfs: return []
    launch_t = min((w['time_ns'] for w in wfs), key=lambda t: abs(t - t_main))
    return [w for w in wfs if abs(w['time_ns'] - launch_t) < 1.0]

atwd_use = near_t_main(atwd_all)
fadc_use = near_t_main(fadc_all)

def best_atwd(wfs):
    if not wfs:
        return None
    return max(wfs, key=lambda w: max((abs(v) for v in w['samples_volt']), default=0))

atwd = best_atwd(atwd_use)
fadc = fadc_use[0] if fadc_use else None

# constrained_layout adjusts axes margins WITHIN the figure rather than
# cropping the figure boundary — so the saved file stays exactly FIGSIZE.
fig, axes = plt.subplots(2, 1, figsize=FIGSIZE, constrained_layout=True)

def draw(ax, w, color, xlim_override=None):
    V = np.asarray(w['samples_volt'])
    dt_ns = w['bin_width_ns']
    pe = V * (dt_ns * 1e-9) * k
    t = w['time_ns'] + np.arange(len(pe)) * dt_ns
    ax.step(t, pe, where='post', lw=LINEWIDTH, color=color)
    if xlim_override is not None:
        ax.set_xlim(*xlim_override)
    else:
        span = (t[-1] + dt_ns) - t[0]
        pad = EDGE_PAD_FRAC * span
        ax.set_xlim(t[0] - pad, t[-1] + dt_ns + pad)
    ax.set_xlabel('time [ns]')
    ax.set_ylabel('charge per sample [PE]')
    ax.set_title(rf"{w['source']}  ({len(pe)} samples, {dt_ns:.2f} ns/bin)   "
                 rf"total {pe.sum():.1f} PE")
    ax.grid(alpha=0.3, lw=0.5)

if atwd:
    draw(axes[0], atwd, ATWD_COLOR, xlim_override=XLIM_ATWD)
if fadc:
    draw(axes[1], fadc, FADC_COLOR, xlim_override=XLIM_FADC)

om_s = '({}, {})'.format(hit['om'][0], hit['om'][1])
fig.suptitle(
    f"{DATA_SOURCE_LABEL}\n"
    rf"run {hit['event']['run_id']}\,\, event {hit['event']['event_id']}\,\, "
    rf"DOM (string, om) = {om_s}"
)

stem = PLOTS_DIR / f"plot_run{hit['event']['run_id']}_event{hit['event']['event_id']}_DOM{hit['om'][0]}-{hit['om'][1]}-{hit['om'][2]}"
if SAVE_PDF:
    export_to_pdf(fig, f"{stem}.pdf")
    print(f"saved {stem}.pdf")
if SAVE_PNG:
    # also no bbox_inches='tight' — file stays exactly FIGSIZE
    fig.savefig(f"{stem}.png", dpi=DPI)
    print(f"saved {stem}.png")

sidecar = Path(f"{stem}.txt")
sidecar.write_text(
    f"Saved waveform plot\n"
    f"-------------------\n"
    f"Data source       : {DATA_SAMPLE}\n"
    f"Run / event       : {hit['event']['run_id']} / {hit['event']['event_id']}\n"
    f"DOM (string,om,pmt): {hit['om']}\n"
    f"HLC launch t_main : {t_main:.1f} ns\n"
    f"PMT HV / gain     : {hit['pmt_hv_volts']:.0f} V / {hit['pmt_gain']:.2e}\n"
    + (f"ATWD ch{atwd['channel']} total : {(np.asarray(atwd['samples_volt'])*atwd['bin_width_ns']*1e-9*k).sum():.1f} PE\n"
       if atwd else "")
    + (f"FADC total        : {(np.asarray(fadc['samples_volt'])*fadc['bin_width_ns']*1e-9*k).sum():.1f} PE\n"
       if fadc else "")
)
print(f"saved {sidecar.name}")

plt.show()